# Übungsblatt: Normalformen in Theorie und Praxis

*Datenbanken · 3. Semester B.Sc. Informatik*

| | |
|---|---|
| **Bearbeitungszeit** | ca. 1,5–2 Stunden (29 Punkte + 4 Bonuspunkte) |
| **Umgebung** | Google Colab oder lokal mit Jupyter (Python ab 3.10). Benötigte Pakete werden automatisch installiert. |
| **Abgabe** | dieses Notebook (`.ipynb`) mit Ihren Lösungen und Antworten |

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Thomarius/datenbanken-uebungen/blob/main/normalformen/Uebungsblatt_Normalformen.ipynb)

> **Zuerst speichern, dann arbeiten**
>
> - **In Google Colab:** Wählen Sie *Datei → Kopie in Drive speichern* und arbeiten Sie in dieser Kopie weiter (sie öffnet sich in einem neuen Tab und wird automatisch gespeichert). Arbeiten Sie nicht in der ursprünglich geöffneten Fassung: Deren Änderungen gehen beim Schließen des Tabs verloren.
> - **Lokal in Jupyter oder VS Code:** Speichern Sie die Datei regelmäßig (*Strg + S*).
> - **Abgabe:** In Colab *Datei → Herunterladen → .ipynb herunterladen*, lokal die gespeicherte `.ipynb`-Datei. Laden Sie diese Datei in Moodle hoch.
> - Setzt Colab die Laufzeit nach längerer Pause zurück, bleibt Ihr Code erhalten. Führen Sie dann einfach *Laufzeit → Alle ausführen* aus.

**So arbeiten Sie mit diesem Notebook**

- Führen Sie die Zellen der Reihe nach aus. Code-Lücken sind mit `# Ihre Lösung` markiert.
- Ändern Sie die Namen der vorgegebenen Funktionen und die Kennzeichnung `# Abgabe …` in der ersten Zeile der Lösungszellen nicht.
- Schreiben Sie Ihre Antworten in die Zellen **Antwort …** (Doppelklick zum Bearbeiten).
- Nach jeder Teilaufgabe gibt es **Selbsttests** mit kleinen Beispielen. Sie zeigen, ob Ihr Ergebnis für *diese Beispiele* stimmt. Bewertet wird mit weiteren, verdeckten Tests und anderen Daten: Ihre Lösung muss für beliebige Relationen funktionieren, nicht nur für die Beispiele.
- **Vor der Abgabe:** *Laufzeit → Neu starten und alle ausführen* und prüfen, dass keine Zelle mit einem Fehler abbricht.

## Vorbereitung

In [ ]:
# Setup – bitte nicht verändern
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("duckdb") is None:  # z. B. in einer frischen Colab-Umgebung
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)

from itertools import combinations

import duckdb

FD = tuple[frozenset[str], frozenset[str]]


def fd(links: str, rechts: str) -> FD:
    """Erzeugt eine FD aus zwei durch Leerzeichen getrennten Attributlisten: fd("a b", "c") ist {a, b} → {c}."""
    return frozenset(links.split()), frozenset(rechts.split())


def sortiert(mengen) -> list[frozenset[str]]:
    """Sortiert Attributmengen nach Größe, dann alphabetisch."""
    return sorted((frozenset(m) for m in mengen), key=lambda m: (len(m), sorted(m)))


ATTRIBUTE = frozenset(
    "matrnr name studiengang kursnr kursname trainer trainer_email halle anmeldedatum bezahlt".split()
)
CSV_DATEI = "kursanmeldungen.csv"

In [ ]:
# Hilfsfunktion für die Selbsttests – bitte nicht verändern
def selbsttest(beschreibung, berechnung, erwartet):
    """Vergleicht Ihr Ergebnis für ein Beispiel mit dem erwarteten Ergebnis."""
    try:
        ergebnis = berechnung()
    except NotImplementedError:
        print(f"·  {beschreibung}: noch nicht implementiert")
        return
    except Exception as fehler:
        print(f"✗  {beschreibung}: {type(fehler).__name__}: {fehler}")
        return
    if ergebnis == erwartet:
        print(f"✓  {beschreibung}: Ergebnis stimmt für dieses Beispiel")
    else:
        print(f"✗  {beschreibung}\n     Ihr Ergebnis: {ergebnis}\n     erwartet:     {erwartet}")

Die Daten des Hochschulsports als CSV-Datei anlegen:

In [ ]:
%%writefile kursanmeldungen.csv
matrnr,name,studiengang,kursnr,kursname,trainer,trainer_email,halle,anmeldedatum,bezahlt
1112233,Mia Becker,Informatik,K01,Yoga,sberg,s.berg@hochschulsport.example,Halle A,2026-10-01,true
1112233,Mia Becker,Informatik,K03,Volleyball,mkoch,m.koch@hochschulsport.example,Halle B,2026-10-02,true
1112244,Noah Wagner,Wirtschaftsinformatik,K02,Bouldern,jlang,j.lang@hochschulsport.example,Kletterhalle,2026-10-01,true
1112244,Noah Wagner,Wirtschaftsinformatik,K04,Badminton,mkoch,m.koch@hochschulsport.example,Halle A,2026-10-05,false
1112255,Emma Schulz,Informatik,K01,Yoga,sberg,s.berg@hochschulsport.example,Halle A,2026-10-02,true
1112255,Emma Schulz,Informatik,K02,Bouldern,jlang,j.lang@hochschulsport.example,Kletterhalle,2026-10-03,true
1112255,Emma Schulz,Informatik,K05,Pilates,sberg,s.berg@hochschulsport.example,Halle B,2026-10-07,false
1112266,Ben Hoffmann,Data Science,K03,Volleyball,mkoch,m.koch@hochschulsport.example,Halle B,2026-10-04,true
1112277,Lina Fischer,Informatik,K02,Bouldern,jlang,jlang@hochschulsport.example,Kletterhalle,2026-10-04,true
1112277,Lina Fischer,Informatik,K04,Badminton,mkoch,m.koch@hochschulsport.example,Halle A,2026-10-06,true
1112288,Paul Krause,Elektrotechnik,K03,Volleyball,mkoch,m.koch@hochschulsport.example,Halle B,2026-10-05,false
1112288,Paul Krause,Elektrotechnik,K04,Badminton,mkoch,m.koch@hochschulsport.example,Halle A,2026-10-06,true
1112299,Sofia Neumann,Data Science,K01,Yoga,sberg,s.berg@hochschulsport.example,Halle A,2026-10-03,true
1112299,Sofia Neumann,Data Science,K02,Bouldern,jlang,j.lang@hochschulsport.example,Kletterhalle,2026-10-08,false
1112299,Sofia Neumann,Data Science,K05,Pilates,sberg,s.berg@hochschulsport.example,Halle B,2026-10-09,true
1112233,Mia Becker,Informatik,K05,Pilates,sberg,s.berg@hochschulsport.example,Halle B,2026-10-10,true
1112244,Noah Wagner,Wirtschaftsinformatik,K01,Yoga,sberg,s.berg@hochschulsport.example,Halle A,2026-10-11,true
1112288,Paul Krause,Elektrotechnik,K02,Bouldern,jlang,j.lang@hochschulsport.example,Kletterhalle,2026-10-12,true

## Szenario: Kursanmeldungen im Hochschulsport

Der Hochschulsport verwaltet die Anmeldungen zu seinen Kursen bisher in einer einzigen Tabelle, die regelmäßig als CSV-Datei exportiert wird. Ein Ausschnitt:

| matrnr | name | studiengang | kursnr | kursname | trainer | trainer_email | halle | anmeldedatum | bezahlt |
|---|---|---|---|---|---|---|---|---|---|
| 1112233 | Mia Becker | Informatik | K01 | Yoga | sberg | s.berg@… | Halle A | 2026-10-01 | true |
| 1112233 | Mia Becker | Informatik | K03 | Volleyball | mkoch | m.koch@… | Halle B | 2026-10-02 | true |
| 1112244 | Noah Wagner | Wirtschaftsinformatik | K02 | Bouldern | jlang | j.lang@… | Kletterhalle | 2026-10-01 | true |
| 1112244 | Noah Wagner | Wirtschaftsinformatik | K04 | Badminton | mkoch | m.koch@… | Halle A | 2026-10-05 | false |
| 1112266 | Ben Hoffmann | Data Science | K03 | Volleyball | mkoch | m.koch@… | Halle B | 2026-10-04 | true |

Aus Gesprächen mit dem Hochschulsport kennen Sie folgende fachliche Regeln:

- Studierende werden über ihre Matrikelnummer identifiziert. Zu jeder Matrikelnummer gehören genau ein Name und ein Studiengang.
- Jeder Kurs hat eine Kursnummer, einen Namen und genau eine Trainerin bzw. einen Trainer. Ein Kurs findet immer in derselben Halle statt.
- Trainerinnen und Trainer werden über ein Kürzel (`trainer`) identifiziert und haben genau eine dienstliche E-Mail-Adresse.
- Studierende können sich für beliebig viele Kurse anmelden, für jeden Kurs aber nur einmal. Zu jeder Anmeldung werden das Anmeldedatum und der Zahlungsstatus gespeichert.
- Namen sind nicht eindeutig: Zwei Studierende können gleich heißen, und in einer Halle finden mehrere Kurse statt.

---
## Aufgabe 1: Theorie (ca. 30 Min., 12 Punkte)

**a) (3 P.)** Geben Sie alle funktionalen Abhängigkeiten (FDs) an, die sich aus den fachlichen Regeln ergeben. Fassen Sie FDs mit gleicher linker Seite zusammen.

*Format:* eine FD pro Zeile (als Liste) in der Form `X → Y`, nummeriert mit F1, F2, …. X und Y sind durch Kommas getrennte Attributnamen, geschrieben wie die Spaltennamen der Tabelle (z. B. `trainer_email`). Statt `→` können Sie auch `->` tippen. Beispiel für das Format (keine FD dieser Aufgabe): `F1: isbn, auflage → erscheinungsjahr`

**Antwort 1a**

- F1: … → …
- F2: … → …

**b) (3 P.)** Berechnen Sie die Attributhüllen {matrnr}⁺, {kursnr}⁺ und {matrnr, kursnr}⁺ mit dem Algorithmus aus der Vorlesung. Bestimmen Sie damit alle Schlüsselkandidaten und begründen Sie, warum es keine weiteren gibt. Ist {matrnr, kursnr, name} ein Schlüsselkandidat?

**Antwort 1b**

*Ihre Antwort*

**c) (2 P.)** Nennen Sie anhand der Daten je ein konkretes Beispiel für eine Änderungs-, eine Einfüge- und eine Löschanomalie.

**Antwort 1c**

*Ihre Antwort*

**d) (1 P.)** In welcher Normalform (1NF, 2NF oder 3NF) ist die Ausgangsrelation höchstens? Begründen Sie kurz.

**Antwort 1d**

*Ihre Antwort*

**e) (3 P.)** Zerlegen Sie die Relation schrittweise in die 2NF und dann in die 3NF. Geben Sie für jede Relation die Attribute an, kennzeichnen Sie Primärschlüssel und Fremdschlüssel und begründen Sie mit der Faustregel aus der Vorlesung, warum Ihre Zerlegung verlustfrei ist.

**Antwort 1e**

*Ihre Antwort*

---
## Aufgabe 2: Normalisierung programmieren (ca. 40 Min., 12 Punkte)

Eine FD ist ein Paar zweier `frozenset`s; `fd("a b", "c")` erzeugt {a, b} → {c}.

**a) (4 P.)** Implementieren Sie `huelle(x, fds)` und `ist_superschluessel(x, attribute, fds)`.

In [ ]:
# Abgabe 2a – Ihre Lösung in dieser Zelle
def huelle(x, fds):
    """Berechnet die Attributhülle X⁺ von x bezüglich fds. Die Eingabe x darf nicht verändert werden."""
    # Ihre Lösung
    raise NotImplementedError("Aufgabe 2a")


def ist_superschluessel(x, attribute, fds):
    """True genau dann, wenn x alle Attribute der Relation bestimmt."""
    # Ihre Lösung
    raise NotImplementedError("Aufgabe 2a")

In [ ]:
# Selbsttests 2a
selbsttest("Hülle ohne FDs", lambda: huelle({"a"}, []), {"a"})
selbsttest("Hülle über eine Kette a → b → c", lambda: huelle({"a"}, [fd("a", "b"), fd("b", "c")]), {"a", "b", "c"})
selbsttest("Hülle mit zusammengesetzter linker Seite", lambda: huelle({"a", "b"}, [fd("a b", "c"), fd("c", "d")]), {"a", "b", "c", "d"})
selbsttest("Superschlüssel {a} in R(a, b, c)", lambda: ist_superschluessel({"a"}, frozenset("abc"), [fd("a", "b"), fd("b", "c")]), True)
selbsttest("Kein Superschlüssel {b} in R(a, b, c)", lambda: ist_superschluessel({"b"}, frozenset("abc"), [fd("a", "b"), fd("b", "c")]), False)

**b) (4 P.)** Implementieren Sie `schluesselkandidaten(attribute, fds)`. Ein einfaches Verfahren genügt: Prüfen Sie alle Teilmengen der Größe nach (`combinations(sorted(attribute), k)`) und überspringen Sie Obermengen bereits gefundener Kandidaten. Geben Sie das Ergebnis mit `sortiert(…)` zurück.

In [ ]:
# Abgabe 2b – Ihre Lösung in dieser Zelle
def schluesselkandidaten(attribute, fds):
    """Liefert alle Schlüsselkandidaten (minimale Superschlüssel), sortiert mit sortiert()."""
    # Ihre Lösung
    raise NotImplementedError("Aufgabe 2b")

In [ ]:
# Selbsttests 2b
selbsttest("Ein Kandidat", lambda: schluesselkandidaten(frozenset("abc"), [fd("a", "b"), fd("b", "c")]), [frozenset("a")])
selbsttest("Zwei Kandidaten bei ab → c, c → a", lambda: schluesselkandidaten(frozenset("abc"), [fd("a b", "c"), fd("c", "a")]),
           [frozenset("ab"), frozenset("bc")])
selbsttest("Ohne FDs: alle Attribute", lambda: schluesselkandidaten(frozenset("abc"), []), [frozenset("abc")])

**c) (4 P.)** Tragen Sie Ihre FDs aus Aufgabe 1a in `HOCHSCHULSPORT_FDS` ein, z. B. wird aus `isbn, auflage → erscheinungsjahr` der Eintrag `fd("isbn auflage", "erscheinungsjahr")`. Führen Sie dann die Zelle darunter aus. Stimmen die Ergebnisse mit Ihren Antworten aus 1b überein? Wie viele Teilmengen muss Ihr Verfahren bei 10 Attributen im schlimmsten Fall prüfen? Welche Beobachtung aus 1b würde die Suche stark beschleunigen?

In [ ]:
# Abgabe 2c – Ihre FDs aus Aufgabe 1a
HOCHSCHULSPORT_FDS = [
    # z. B. fd("matrnr", "name"),
]

In [ ]:
try:
    for x in ["matrnr", "kursnr", "matrnr kursnr"]:
        print(f"{{{x}}}⁺ =", sorted(huelle(set(x.split()), HOCHSCHULSPORT_FDS)))
    print("Schlüsselkandidaten:", schluesselkandidaten(ATTRIBUTE, HOCHSCHULSPORT_FDS))
except NotImplementedError as fehler:
    print("Noch nicht implementiert:", fehler)

**Antwort 2c**

*Ihre Antwort*

**d) (Bonus, +2 P.)** Implementieren Sie `partielle_abhaengigkeiten(attribute, fds)`. Die Funktion liefert alle Paare (S, A), bei denen das Nichtprimattribut A schon von einer echten, nichtleeren Teilmenge S eines Schlüsselkandidaten abhängt (Verletzung der 2NF), sortiert nach `(sorted(S), A)`.

In [ ]:
# Abgabe 2d – Ihre Lösung in dieser Zelle (Bonus)
def partielle_abhaengigkeiten(attribute, fds):
    """Liefert alle Verletzungen der 2NF als Paare (Teilmenge eines Schlüssels, Nichtprimattribut)."""
    # Ihre Lösung
    raise NotImplementedError("Aufgabe 2d")

In [ ]:
# Selbsttests 2d
selbsttest("Eine partielle Abhängigkeit", lambda: partielle_abhaengigkeiten(frozenset("abcd"), [fd("a b", "c"), fd("a", "d")]),
           [(frozenset("a"), "d")])
selbsttest("Primattribute zählen nicht", lambda: partielle_abhaengigkeiten(frozenset("abc"), [fd("a b", "c"), fd("c", "a")]), [])

---
## Aufgabe 3: FDs in echten Daten prüfen (ca. 20 Min., 5 Punkte)

FDs sind Fachwissen. Ob sich die Daten daran halten, ist eine andere Frage. Diese prüfen wir mit SQL in DuckDB, einer Datenbank, die direkt im Notebook läuft.

### Kurzeinführung: SQL im Notebook

- `con = duckdb.connect()` erzeugt eine neue, leere Datenbank im Arbeitsspeicher.
- `con.execute("…")` führt eine SQL-Anweisung aus. Mit `.fetchall()` erhalten Sie das Ergebnis als Python-Liste von Tupeln, eines je Zeile.
- `con.sql("…").show()` zeigt das Ergebnis einer Abfrage als Tabelle an, praktisch zum Ausprobieren.
- SQL steht als Text in Anführungszeichen. Texte *innerhalb* von SQL stehen in einfachen Anführungszeichen (`'Rose'`). Mehrzeiliges SQL steht zwischen dreifachen Anführungszeichen, wie im Beispiel unten.
- Mit einem f-String (`f"… {variable} …"`) setzen Sie Python-Werte wie Tabellen- oder Spaltennamen in den SQL-Text ein. `", ".join(liste)` macht aus einer Liste von Spaltennamen einen kommagetrennten Text.
- `read_csv('datei.csv')` liest eine CSV-Datei direkt als Tabelle.

Führen Sie das Beispiel aus und probieren Sie gern eigene Abfragen:

In [ ]:
# Beispiel: SQL mit DuckDB
con_bsp = duckdb.connect()
con_bsp.execute("CREATE TABLE pflanzen (art TEXT, standort TEXT, hoehe INTEGER)")
con_bsp.execute("""
    INSERT INTO pflanzen VALUES
      ('Rose', 'Beet', 60), ('Rose', 'Kübel', 40), ('Tulpe', 'Beet', 30), ('Tulpe', 'Beet', 35)
""")

# Ergebnis als Tabelle anzeigen
con_bsp.sql("SELECT * FROM pflanzen").show()

# Ergebnis in Python weiterverarbeiten: Liste von Tupeln
zeilen = con_bsp.execute("SELECT art, COUNT(*) AS anzahl FROM pflanzen GROUP BY art ORDER BY art").fetchall()
print(zeilen)

# Anzahl verschiedener Werte je Gruppe, und Gruppen mit HAVING filtern
print(con_bsp.execute("SELECT art, COUNT(DISTINCT standort) FROM pflanzen GROUP BY art ORDER BY art").fetchall())
print(con_bsp.execute("SELECT art FROM pflanzen GROUP BY art HAVING COUNT(*) > 1 ORDER BY art").fetchall())

# Spaltennamen aus einer Python-Liste einsetzen
spalten = ["art", "standort"]
print(con_bsp.execute(f"SELECT DISTINCT {', '.join(spalten)} FROM pflanzen ORDER BY {', '.join(spalten)}").fetchall())

# CSV-Datei direkt abfragen
con_bsp.sql(f"SELECT name, kursname, halle FROM read_csv('{CSV_DATEI}') LIMIT 3").show()

**a) (3 P.)** Implementieren Sie `fd_verletzungen(con, tabelle, links, rechts)`. Die Funktion soll per SQL alle Wertekombinationen der linken Seite finden, zu denen mehr als ein verschiedener Wert der rechten Seite vorkommt, als Liste von Tupeln, aufsteigend sortiert. Eine leere Liste bedeutet: Die Daten widersprechen der FD nicht.

In [ ]:
# Abgabe 3a – Ihre Lösung in dieser Zelle
def fd_verletzungen(con, tabelle, links, rechts):
    """Liefert die Werte von `links` (als Tupel, sortiert), zu denen es mehrere Werte von `rechts` gibt."""
    # Ihre Lösung
    raise NotImplementedError("Aufgabe 3a")

In [ ]:
# Selbsttests 3a
con_test = duckdb.connect()
con_test.execute("CREATE TABLE t (x INTEGER, y TEXT, z TEXT)")
con_test.execute("INSERT INTO t VALUES (1, 'p', 'u'), (1, 'p', 'v'), (2, 'q', 'u'), (2, 'r', 'u'), (3, 's', 'w')")
selbsttest("FD y → x gilt", lambda: fd_verletzungen(con_test, "t", ["y"], "x"), [])
selbsttest("FD x → y verletzt", lambda: fd_verletzungen(con_test, "t", ["x"], "y"), [(2,)])
selbsttest("Zusammengesetzte linke Seite", lambda: fd_verletzungen(con_test, "t", ["x", "y"], "z"), [(1, "p")])

**b) (2 P.)** Laden Sie die CSV-Datei in eine Tabelle `roh` und prüfen Sie jede FD aus 1a an den Daten. Welche FD ist verletzt? Welche Zeile ist vermutlich fehlerhaft und wie lautet der korrekte Wert? Welche Tabelle Ihres 3NF-Schemas aus 1e hätte diesen Fehler verhindert, und warum?

In [ ]:
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE TABLE roh AS SELECT * FROM read_csv('{CSV_DATEI}')")
# Prüfen Sie hier jede FD aus 1a, z. B.:
# print(fd_verletzungen(con, "roh", ["matrnr"], "name"))

**Antwort 3b**

*Ihre Antwort*

---
## Bonus: KI als Datenmodelliererin? (ca. 15 Min., +2 Punkte)

Sprachmodelle können funktionale Abhängigkeiten vorschlagen. Aber woher wissen sie, was gilt? Prüfen Sie das mit Ihren eigenen Werkzeugen.

1. Geben Sie einem Sprachmodell Ihrer Wahl **nur den Tabellenausschnitt** aus dem Szenario und bitten Sie es, alle funktionalen Abhängigkeiten anzugeben. Wiederholen Sie das in einem **neuen Chat mit Tabellenausschnitt und fachlichen Regeln**.
2. Tragen Sie beide Antworten unten als `KI_FDS_OHNE_REGELN` und `KI_FDS_MIT_REGELN` ein, im Format aus 2c.
3. Implementieren Sie `folgt_aus(fd_, fds)` und ergänzen Sie die Prüfung:
   - Folgt jede KI-FD aus Ihren FDs aus 1a, und folgt umgekehrt jede Ihrer FDs aus den KI-FDs? *Tipp:* X → Y folgt aus F genau dann, wenn Y ⊆ X⁺ bezüglich F.
   - Welche KI-FDs werden von den echten Daten (`roh`) widerlegt? Nutzen Sie `fd_verletzungen`.
4. Dokumentieren Sie in der Antwortzelle Ihre beiden Prompts und erklären Sie mindestens einen Unterschied zwischen den KI-FDs und Ihren FDs: Woran lag er?

In [ ]:
# Bonus – FDs des Sprachmodells (Format wie in 2c)
KI_FDS_OHNE_REGELN = [
    # fd("…", "…"),
]
KI_FDS_MIT_REGELN = [
    # fd("…", "…"),
]

In [ ]:
# Bonus – Prüfung mit den eigenen Werkzeugen
def folgt_aus(fd_, fds):
    """True, wenn die FD fd_ = (X, Y) aus der FD-Menge fds folgt."""
    # Ihre Lösung
    raise NotImplementedError("Bonus")


con_bonus = duckdb.connect()
con_bonus.execute(f"CREATE OR REPLACE TABLE roh AS SELECT * FROM read_csv('{CSV_DATEI}')")

for titel, ki_fds in [("ohne Regeln", KI_FDS_OHNE_REGELN), ("mit Regeln", KI_FDS_MIT_REGELN)]:
    print(f"--- KI-FDs {titel} ---")
    # Für jede KI-FD: Folgt sie aus HOCHSCHULSPORT_FDS? Wird sie von den Daten widerlegt?
    # Umgekehrt: Welche FDs aus HOCHSCHULSPORT_FDS folgen nicht aus den KI-FDs?
    pass

**Antwort Bonus**

*Ihre Antwort*

---
**Vor der Abgabe:** *Laufzeit → Neu starten und alle ausführen* und prüfen, dass keine Zelle mit einem Fehler abbricht. Dann das Notebook als `.ipynb` herunterladen (in Colab: *Datei → Herunterladen → .ipynb herunterladen*) und in Moodle hochladen.